Write a function that reads an xlsx file. The file is expected to have a sheet called "stream_data" with headers
ts	delta time	keithley2602_1.smua.v	keithley2602_1.smub.v	keithley2602_1.smua.i	keithley2602_1.smub.i	keithley2602_1.smua.setv	keithley2602_1.smub.setv
And another sheet called "notes" which has a information in cell B2 that looks something like "HZOFeFETv4_SX22A-top_DG. Bottom gate floating. 6430 on top gate, 2602A C1 for top devices, C4 for bottom devices, 2602B on C4 or C1.". You do not need to verify that the xlsx file is structured this way.
The sample ID is "HZOFeFETv4_SX22#-###". Pull this information out of the file as well. Finally, that string also has information about which gate is floating (bottom or top). Pull that information as well.

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tpo_tools import twinx, fig2clipboard, linear_fit_n_variables
import time

def read_measurement_xlsx(path: str):
    """
    Read measurement data from an xlsx file and extract metadata from notes!B2.

    Returns:
        dict with:
            - stream_data: DataFrame from sheet "stream_data"
            - notes_b2: raw text from notes!B2
            - sample_id: extracted sample ID (or None)
            - floating_gate: "top", "bottom", or None
    """
    # Read the stream data sheet
    stream_data = pd.read_excel(path, sheet_name="stream_data").iloc[1:-1].reset_index(drop=True)

    # Read notes sheet and pull cell B2 (row index 1, col index 1)
    notes_df = pd.read_excel(path, sheet_name="notes", header=None)
    notes_b2 = str(notes_df.iloc[1, 1]) if not pd.isna(notes_df.iloc[1, 1]) else ""

    # Extract sample ID
    # Preferred pattern: HZOFeFETv4_SX22#-### (e.g., HZOFeFETv4_SX22A-123)
    m = re.search(r"HZOFeFETv4_SX16[A-Za-z0-9]-\d{3}", notes_b2)
    if m:
        sample_id = m.group(0)
    else:
        # Fallback for variants like HZOFeFETv4_SX22A-top_DG
        m2 = re.search(r"HZOFeFETv4_SX16[^\s.]*", notes_b2)
        sample_id = m2.group(0) if m2 else None

    # Extract which gate is floating
    lower_text = notes_b2.lower()
    if "bottom gate floating" in lower_text:
        floating_gate = "bottom"
    elif "top gate floating" in lower_text:
        floating_gate = "top"
    else:
        floating_gate = None

    # Replace sentinel overflow values with NaN
    stream_data = stream_data.mask(stream_data[stream_data.columns] > 9e37)

    return {
        "stream_data": stream_data,
        "notes_b2": notes_b2,
        "sample_id": sample_id,
        "floating_gate": floating_gate,
    }

In [ ]:
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=["r", "r", "b", "b", 'g','g'],
    linestyle=["--", "-", "--", "-", '-','-'],
)

for letter in 'ABCDEFGHIJKLMNOP':
    j = -1
    fig,axes = plt.subplots(2,2,figsize=(9,5))
    axes:list[list[plt.Axes]]
    
    axes[0,0].set_title(f'HZOFeFETv4_SX22{letter}, floating top gate')
    axes[0,1].set_title(f'HZOFeFETv4_SX22{letter}, floating bottom gate')
    for ax in axes.reshape(-1):
        ax.set_ylim(9e-13, 2e-5)
        ax.set_xlim(-1.8,1.8)
    axes[1,0].set_xlabel('$V_g$ (V)')
    axes[1,1].set_xlabel('$V_g$ (V)')
    axes[0,0].set_ylabel('$I$ (A), top device')
    axes[1,0].set_ylabel('$I$ (A), btm device')

    for i,f in enumerate(glob('./*.xlsx')):
        data = read_measurement_xlsx(f)
        if data['sample_id'] is None:
            continue

        if f'SX22{letter}-' not in data['sample_id']:
            continue
        # if 'top' not in data['floating_gate']:
        #     continue
        j += 1


        x = data['stream_data']['keithley6430_1.v']
        y = np.abs((data['stream_data']['keithley2602_2.smua.i'] - data['stream_data']['keithley2602_2.smub.i'])/2/np.sign(data['stream_data']['keithley2602_2.smua.v']))
        y2 = np.abs(data['stream_data']['keithley6430_1.i'])
        n = len(x)
        mid = n // 2

        xv = x.to_numpy()
        yv = y.to_numpy()

        X = np.full((n, 4), np.nan)
        Y = np.full((n, 4), np.nan)

        col = 0
        for s, e in [(0, mid), (mid, n)]:
            q = s + max(1, (e - s) // 4)  # first quarter endpoint
            X[s:q, col] = xv[s:q]
            Y[s:q, col] = yv[s:q]
            col += 1
            X[q:e, col] = xv[q:e]
            Y[q:e, col] = yv[q:e]
            col += 1

        ax:plt.Axes = axes.T.reshape(-1)[j]
        ax.plot(X,Y, label=['1st $I_d$, 1st 1/4',
                            '1st $I_d$, remainder',
                            '2nd $I_d$, 1st 1/4',
                            '2nd $I_d$, remainder'])
        ax.plot(x,y2, label='$I_g$')
        ax.set_yscale('log')
        ax.grid()
        if j == 0:
            ax.legend(loc='upper left')
        if 'top' not in data['floating_gate']:
            ax.set_ylabel('')
            ax.set_yticklabels([])
        if 'top' in data['sample_id']:
            ax.set_xlabel('')
            ax.set_xticklabels([])
    fig.tight_layout()
    # fig2clipboard(fig, 350)
    print(letter)
    # time.sleep(1)
    # plt.close()
    break

In [ ]:
for letter in 'ABCDEFGHIJKLMNOP':
    j = -1
    fig,axes = plt.subplots(2,1,figsize=(5,5))
    axes:list[plt.Axes]

    axes[0].set_title(f'HZOFeFETv4_SX16{letter} (BG)')
    for ax in axes.reshape(-1):
        ax.set_ylim(9e-13, 2e-5)
        ax.set_xlim(-1.8,1.8)
    axes[1].set_xlabel('$V_g$ (V)')
    axes[0].set_ylabel('$I$ (A), top device')
    axes[1].set_ylabel('$I$ (A), btm device')

    for i,f in enumerate(glob('./*.xlsx')):
        data = read_measurement_xlsx(f)
        if data['sample_id'] is None:
            continue

        if f'SX16{letter}-' not in data['sample_id']:
            continue
        # if 'top' not in data['floating_gate']:
        #     continue
        j += 1


        x = data['stream_data']['keithley6430_1.v']
        y = np.abs((data['stream_data']['keithley2602_2.smua.i'] - data['stream_data']['keithley2602_2.smub.i'])/2/np.sign(data['stream_data']['keithley2602_2.smua.v']))
        y2 = np.abs(data['stream_data']['keithley6430_1.i'])
        n = len(x)
        mid = n // 2

        xv = x.to_numpy()
        yv = y.to_numpy()

        X = np.full((n, 4), np.nan)
        Y = np.full((n, 4), np.nan)

        col = 0
        for s, e in [(0, mid), (mid, n)]:
            q = s + max(1, (e - s) // 4)  # first quarter endpoint
            X[s:q, col] = xv[s:q]
            Y[s:q, col] = yv[s:q]
            col += 1
            X[q:e, col] = xv[q:e]
            Y[q:e, col] = yv[q:e]
            col += 1

        ax:plt.Axes = axes[j]
        ax.plot(X,Y, label=['1st $I_d$, 1st 1/4',
                            '1st $I_d$, remainder',
                            '2nd $I_d$, 1st 1/4',
                            '2nd $I_d$, remainder'])
        ax.plot(x,y2, label='$I_g$')
        ax.set_yscale('log')
        ax.grid()
        if j == 0:
            ax.legend(loc='upper left')
        # if 'top' not in data['floating_gate']:
        #     ax.set_ylabel('')
        #     ax.set_yticklabels([])
        # if 'top' in data['sample_id']:
        #     ax.set_xlabel('')
        #     ax.set_xticklabels([])
    fig.tight_layout()
    fig2clipboard(fig, 350)
    print(letter)
    time.sleep(1)
    plt.close()
    # break

## 2026/04/02 and 2026/04/03 data on dummy samples.

These are real transistors, but they are not on a window for us to image. They are all to the right of chips A and B. Good for debugging.

In [ ]:
notes_col = "notes_1.text"
GV = 'keithley6430_2.v'
GVset = 'keithley6430_2.setv'
GI = 'keithley6430_2.i'
DV = 'keithley2602_3.smua.v'
DVset = 'keithley2602_3.smua.setv'
DI = 'keithley2602_3.smua.i'

def read_transistor_measurement_xlsx(path: str, sheet_name=0) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet_name)

    if notes_col in df.columns:
        df[notes_col] = df[notes_col].astype(str).str.split("\n").str[-1].str.strip()

    return df

file_path = r'2026-04-03\2026-04-03_131153_measurement.xlsx'
df = read_transistor_measurement_xlsx(file_path)

for note in df[notes_col].unique():
    if note == 'Device -':
        continue
    elif note == 'Device 10':
        subset = df[df[notes_col] == note].iloc[-280-32:-140-32-70]
    else:
        subset = df[df[notes_col] == note].iloc[-140:-70]
    subset.reset_index(inplace=True)

    title = note
    if np.any(np.abs(subset[GV] - subset[GVset]) > 1e-2):
        title = title + ', G compliance'
    if np.any(np.abs(subset[DI]) >= 1e-5):
        title = title + ', D compliance'
    fig, ax = plt.subplots(figsize=(10, 6))
    ax2 = twinx(ax)
    ax3 = twinx(ax, color='b', pos=1.1)
    ax.set_title(title)
    ax.set_xlabel('Index')
    ax.set_ylabel('$V_g$ (V)')
    ax2.set_ylabel('$I_g$ (nA)')
    ax3.set_ylabel('$I_d$ (uA)')
    ax.set_ylim(-2.2,2.2)
    ax2.set_ylim(-1.1,1.1)
    ax3.set_ylim(-11,11)

    ax.plot(subset.index, subset[GV], 'k-', label='actual V')
    ax.plot(subset.index, subset[GVset], 'g--', label='set V', lw=4, alpha=0.6)
    ax2.plot(subset.index, subset[GI]*1e9, 'r-')
    ax3.plot(subset.index, subset[DI]*1e6, 'b-')
    ax.grid()
    ax.legend(loc='lower left')
    # break

## Gate leakage measurements 2026/04/07

In [ ]:
from matplotlib.colors import LogNorm

# path = '2026-04-07\\2026-04-07_174358_measurement.xlsx'
path = '2026-04-07\\2026-04-07_182657_measurement.xlsx'
df = pd.read_excel(path)

R1 = np.nan*np.ones((8,4))
R2 = np.nan*np.ones((8,4))
R  = np.nan*np.ones((8,4))

for i,letter in enumerate('ABCDEFG'):
    for j,device in enumerate(('top', 'btm')):
        _df = df[df['notes_3.text'] == f'SX16{letter} {device}']
        if letter == 'E' and device == 'btm':
            continue

        lo = _df.last_valid_index()-21
        hi = _df.last_valid_index()

        x  = df['keithley6430_2.v'][lo:hi]
        y1 = -df['keithley2602_1.smua.i'][lo:hi]
        y2 = -df['keithley2602_1.smub.i'][lo:hi]
        y  = df['keithley6430_2.i'][lo:hi]

        (m1,_),_,_ = linear_fit_n_variables(y1, x)
        (m2,_),_,_ = linear_fit_n_variables(y2, x)
        (m,_),_,_ = linear_fit_n_variables(y, x)
        print(f'SX16{letter} {device} ({lo},{hi}): R1 = {1/m1*1e-6:0.3f} Mohm, R2 = {1/m2*1e-6:0.3f} Mohm, R = {1/m*1e-6:0.3f} Mohm')
        
        plt.figure()
        plt.title(f'SX16{letter} {device}')
        plt.plot(x,y1)
        plt.plot(x,y2)
        plt.plot(x,y)

        R1[2*(i//4)+j%2, i%4] = 1/m1*1e-6
        R2[2*(i//4)+j%2, i%4] = 1/m2*1e-6
        R [2*(i//4)+j%2, i%4] = 1/m *1e-6

fig,axes = plt.subplots(1,4, figsize=(10,4))
cmap = plt.colormaps.get_cmap('gray')
cmap.set_bad(color='red')  # Set NaNs to red
axes[0].matshow(R1, aspect=0.5, norm=LogNorm(vmin=0.01, vmax=1000), cmap=cmap)
axes[0].set_title('R btwn gate and source')
axes[1].matshow(R2, aspect=0.5, norm=LogNorm(vmin=0.01, vmax=1000), cmap=cmap)
axes[1].set_title('R btwn gate and drain')
im = axes[2].matshow(R , aspect=0.5, norm=LogNorm(vmin=0.01, vmax=1000), cmap=cmap)
axes[2].set_title('R btwn gate and src+drn')
plt.colorbar(im, axes[3], axes[2], aspect=0.25)
axes[3].set_ylabel('resistance (Mohm)')
fig.tight_layout()

## 2026/04/08 FeCap heaters

In [ ]:
path = '2026-04-08\\2026-04-08_123700_measurement.xlsx'

df = pd.read_excel(path)
"FeHZO SX35A top TE\nFor TE (BE), biasing C1 (C2), grounding C3 (C4)\nMFM, already annealed at GA Tech"

for i,letter in enumerate('ABCD'):
    for electrode in ('TE','BE'):
        lbl = f'SX35{letter} top {electrode}'
        _df = df[df['notes_3.text'].str.contains(lbl)][-11:]

        x  = _df['keithley6430_2.v']
        y  = _df['keithley6430_2.i']

        (m,_),_,_ = linear_fit_n_variables(y, x)
        print(f'{lbl} {1/m/1e3:0.3f} kohm')
    
    lbl = f'SX35{letter} btm heater'
    _df = df[df['notes_3.text'].str.contains(lbl)][-11:]

    x  = _df['keithley6430_2.v']
    y  = _df['keithley6430_2.i']

    (m,_),_,_ = linear_fit_n_variables(y, x)
    print(f'{lbl} {1/m/1e3:0.3f} kohm')

## 2026/04/09 Measuring DG FeFETs (HZOFeFETv4 SX22) after KOH etch

In [ ]:
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=["r", "r", "b", "b", 'g','g'],
    linestyle=["--", "-", "--", "-", '-','-'],
)

# df = pd.read_excel('2026-04-09\\2026-04-09_144948_measurement.xlsx')
df = pd.read_excel('2026-04-09\\2026-04-10_090658_measurement.xlsx')


for letter in 'ABCDEFGHIJKLMNOP':
    fig,axes = plt.subplots(2,2,figsize=(9,5))
    axes:list[list[plt.Axes]]
    
    axes[0,0].set_title(f'HZOFeFETv4_SX22{letter}, floating top gate')
    axes[0,1].set_title(f'HZOFeFETv4_SX22{letter}, floating bottom gate')
    for ax in axes.reshape(-1):
        ax.set_ylim(9e-13, 2e-5)
        ax.set_xlim(-1.8,1.8)
    axes[1,0].set_xlabel('$V_g$ (V)')
    axes[1,1].set_xlabel('$V_g$ (V)')
    axes[0,0].set_ylabel('$I$ (A), top device')
    axes[1,0].set_ylabel('$I$ (A), btm device')

    for device,row_of_axes in zip(('top','btm'),axes):
        for gate,ax in zip(('BG','TG'),row_of_axes):
            _df = df[df['notes_2.text'].str.contains(f'SX22{letter} {device} {gate} only')]

            __df = _df[_df['notes_2.text'].str.contains('redo')]
            if len(__df):
                _df = __df
                # print(f'REDO OF SX22{letter} {device} {gate} only')
                redo = True
            else:
                redo = False

            _df = _df[-140:]

            x = _df['keithley2602_1.smua.v'].to_numpy()
            y = _df['keithley2602_1.smub.i'].to_numpy()
            y2 = _df['keithley2602_1.smua.i'].to_numpy()

            x -= x[-1]
            y -= y[-1]
            y2 -= y2[-1]

            x2 = _df['keithley2602_1.smub.v'].to_numpy()
            x = x[x2 > 0]
            y = y[x2 > 0]
            y2 = y2[x2 > 0]
        
            n = len(x)
            mid = n // 2

            X = np.full((n, 4), np.nan)
            Y = np.full((n, 4), np.nan)

            col = 0
            for s, e in [(0, mid), (mid, n)]:
                q = s + max(1, (e - s) // 4)  # first quarter endpoint
                X[s:q, col] = x[s:q]
                Y[s:q, col] = y[s:q]
                col += 1
                X[q:e, col] = x[q:e]
                Y[q:e, col] = y[q:e]
                col += 1

            ax.plot(X,Y, label=['1st $I_d$, 1st 1/4',
                                '1st $I_d$, remainder',
                                '2nd $I_d$, 1st 1/4',
                                '2nd $I_d$, remainder'])
            ax.plot(x,np.abs(y2), label='$|I_g|$')
            ax.set_yscale('log')
            ax.grid()
            if redo:
                ax.text(0.7,0.85, '100nA sense\nrange $I_g$', transform=ax.transAxes)

            if 'TG' in gate:
                ax.set_ylabel('')
                ax.set_yticklabels([])
            if 'top' in device:
                ax.set_xlabel('')
                ax.set_xticklabels([])
    axes[0,0].legend(loc='upper left')
    fig.tight_layout()
    # fig2clipboard(fig, 350)
    print(letter)
    # time.sleep(1)
    # plt.close()
    break

In [ ]:
df = pd.read_excel('2026-04-09\\2026-04-10_090658_measurement.xlsx')

for letter in 'ABCDEFGHIJKLMNOP':
    for device,row_of_axes in zip(('top','btm'),axes):
        for gate,ax in zip(('BG','TG'),row_of_axes):
            _df = df[df['notes_2.text'].str.contains(f'SX22{letter} {device} {gate} only')]
            __df = _df[_df['notes_2.text'].str.contains('redo')]
            if len(__df):
                _df = __df

            _df = _df[-140:]

            VG = _df['keithley2602_1.smua.v'].to_numpy()
            ID = _df['keithley2602_1.smub.i'].to_numpy()
            IG = _df['keithley2602_1.smua.i'].to_numpy()

            if np.any(np.abs(IG) > 1e-7):
                continue
            print(f'Consider redoing SX22{letter} {device} {gate} only')

## 2026/04/10 retesting SX16 devices with high gate compliance

In [ ]:
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=["r", "r", "b", "b", 'g','g'],
    linestyle=["--", "-", "--", "-", '-','-'],
)

df = pd.read_excel('2026-04-10\\2026-04-10_140859_measurement.xlsx')

for letter in 'ABCDEFG':
    fig,axes = plt.subplots(2,1,figsize=(5,5))
    axes:list[list[plt.Axes]]
    
    axes[0].set_title(f'HZOFeFETv4_SX16{letter} (BG)')
    axes[0].set_xlabel('')
    axes[1].set_xlabel('$V_g$ (V)')
    axes[0].set_ylabel('$I$ (A), top device')
    axes[1].set_ylabel('$I$ (A), btm device')
    axes[0].set_xticklabels([])
    for ax in axes.reshape(-1):
        ax.set_ylim(9e-13, 2e-5)
        ax.set_xlim(-1.8,1.8)

    for device,ax in zip(('top','btm'),axes):
            _df = df[df['notes_2.text'].str.contains(f'SX16{letter} {device}')]

            _df = _df[-140:]

            x = _df['keithley2602_1.smua.v'].to_numpy()
            y = _df['keithley2602_1.smub.i'].to_numpy()
            y2 = _df['keithley2602_1.smua.i'].to_numpy()

            x -= x[-1]
            y -= y[-1]
            y2 -= y2[-1]

            x2 = _df['keithley2602_1.smub.v'].to_numpy()
            x = x[x2 > 0]
            y = y[x2 > 0]
            y2 = y2[x2 > 0]
        
            n = len(x)
            mid = n // 2

            X = np.full((n, 4), np.nan)
            Y = np.full((n, 4), np.nan)

            col = 0
            for s, e in [(0, mid), (mid, n)]:
                q = s + max(1, (e - s) // 4)  # first quarter endpoint
                X[s:q, col] = x[s:q]
                Y[s:q, col] = y[s:q]
                col += 1
                X[q:e, col] = x[q:e]
                Y[q:e, col] = y[q:e]
                col += 1

            ax.plot(X,Y, label=['1st $I_d$, 1st 1/4',
                                '1st $I_d$, remainder',
                                '2nd $I_d$, 1st 1/4',
                                '2nd $I_d$, remainder'])
            ax.plot(x,np.abs(y2), label='$|I_g|$')
            ax.set_yscale('log')
            ax.grid()

    axes[0].legend(loc='upper left')
    fig.tight_layout()
    fig2clipboard(fig, 350)
    print(letter)
    time.sleep(1)
    plt.close()
    # break

## 2026/04/10 testing a few more devices on SX16

In [ ]:
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=["r", "r", "b", "b", 'g','g'],
    linestyle=["--", "-", "--", "-", '-','-'],
)

df = pd.read_excel('2026-04-10\\2026-04-10_143527_measurement.xlsx')

for letter in 'HIJKL':
    fig,axes = plt.subplots(2,1,figsize=(5,5))
    axes:list[list[plt.Axes]]
    
    axes[0].set_title(f'HZOFeFETv4_SX16{letter} (BG)')
    axes[0].set_xlabel('')
    axes[1].set_xlabel('$V_g$ (V)')
    axes[0].set_ylabel('$I$ (A), top device')
    axes[1].set_ylabel('$I$ (A), btm device')
    axes[0].set_xticklabels([])
    for ax in axes.reshape(-1):
        ax.set_ylim(9e-13, 2e-5)
        ax.set_xlim(-1.8,1.8)

    for device,ax in zip(('top','btm'),axes):
            _df = df[df['notes_2.text'].str.contains(f'SX16{letter} {device}')]

            _df = _df[-140:]

            x = _df['keithley2602_1.smua.v'].to_numpy()
            y = _df['keithley2602_1.smub.i'].to_numpy()
            y2 = _df['keithley2602_1.smua.i'].to_numpy()

            x -= x[-1]
            y -= y[-1]
            y2 -= y2[-1]

            x2 = _df['keithley2602_1.smub.v'].to_numpy()
            x = x[x2 > 0]
            y = y[x2 > 0]
            y2 = y2[x2 > 0]
        
            n = len(x)
            mid = n // 2

            X = np.full((n, 4), np.nan)
            Y = np.full((n, 4), np.nan)

            col = 0
            for s, e in [(0, mid), (mid, n)]:
                q = s + max(1, (e - s) // 4)  # first quarter endpoint
                X[s:q, col] = x[s:q]
                Y[s:q, col] = y[s:q]
                col += 1
                X[q:e, col] = x[q:e]
                Y[q:e, col] = y[q:e]
                col += 1

            ax.plot(X,Y, label=['1st $I_d$, 1st 1/4',
                                '1st $I_d$, remainder',
                                '2nd $I_d$, 1st 1/4',
                                '2nd $I_d$, remainder'])
            ax.plot(x,np.abs(y2), label='$|I_g|$')
            ax.set_yscale('log')
            ax.grid()

    axes[0].legend(loc='upper left')
    fig.tight_layout()
    fig2clipboard(fig, 350)
    print(letter)
    time.sleep(1)
    plt.close()
    # break

## 2026/04/10 measurements of HZOFeFETv3 SX14 devices.

I just never did measurements on these devices. I guess I forgot they were here.

In [ ]:
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=["r", "r", "b", "b", 'g','g'],
    linestyle=["--", "-", "--", "-", '-','-'],
)

df = pd.read_excel('2026-04-10\\2026-04-10_154831_measurement.xlsx')


for letter in 'ABCDEFGHIJKLMNOP':
    fig,axes = plt.subplots(2,2,figsize=(9,5))
    axes:list[list[plt.Axes]]
    
    axes[0,0].set_title(f'HZOFeFETv4_SX14{letter}, floating top gate')
    axes[0,1].set_title(f'HZOFeFETv4_SX14{letter}, floating bottom gate')
    for ax in axes.reshape(-1):
        ax.set_ylim(9e-13, 2e-6)
        ax.set_xlim(-1.8,1.8)
    axes[1,0].set_xlabel('$V_g$ (V)')
    axes[1,1].set_xlabel('$V_g$ (V)')
    axes[0,0].set_ylabel('$I$ (A), top device')
    axes[1,0].set_ylabel('$I$ (A), btm device')

    for device,row_of_axes in zip(('top','btm'),axes):
        for gate,ax in zip(('BG','TG'),row_of_axes):
            _df = df[df['notes_2.text'].str.contains(f'SX14{letter} {device} {gate} only')]

            _df = _df[-140:]

            x = _df['keithley2602_1.smua.v'].to_numpy()
            y = _df['keithley2602_1.smub.i'].to_numpy()
            y2 = _df['keithley2602_1.smua.i'].to_numpy()

            x2 = _df['keithley2602_1.smub.setv'].to_numpy()
            x -= np.mean(x[x2 == 0])
            y -= np.mean(y[x2 == 0])
            y2 -= np.mean(y2[x2 == 0])

            x = x[x2 > 0]
            y = y[x2 > 0]
            y2 = y2[x2 > 0]
        
            n = len(x)
            mid = n // 2

            X = np.full((n, 4), np.nan)
            Y = np.full((n, 4), np.nan)

            col = 0
            for s, e in [(0, mid), (mid, n)]:
                q = s + max(1, (e - s) // 4)  # first quarter endpoint
                X[s:q, col] = x[s:q]
                Y[s:q, col] = y[s:q]
                col += 1
                X[q:e, col] = x[q:e]
                Y[q:e, col] = y[q:e]
                col += 1

            ax.plot(X,Y, label=['1st $I_d$, 1st 1/4',
                                '1st $I_d$, remainder',
                                '2nd $I_d$, 1st 1/4',
                                '2nd $I_d$, remainder'])
            ax.axhline(y=1e-10, c='g', ls='--', label='EBIC TIA max')
            ax.plot(x,np.abs(y2), label='$|I_g|$')
            ax.set_yscale('log')
            ax.grid()

            if 'TG' in gate:
                ax.set_ylabel('')
                ax.set_yticklabels([])
            if 'top' in device:
                ax.set_xlabel('')
                ax.set_xticklabels([])
    axes[0,0].legend(loc='upper left')
    fig.tight_layout()
    fig2clipboard(fig, 350)
    print(letter)
    time.sleep(1)
    plt.close()
    # break

## 2026/04/10 FeCap heaters SX35 post KOH etch

In [ ]:
path = '2026-04-10\\2026-04-10_175657_measurement.xlsx'

df = pd.read_excel(path)
"FeHZO SX35A top TE\nFor TE (BE), biasing C1 (C2), grounding C3 (C4)\nMFM, already annealed at GA Tech"

for i,letter in enumerate('ABCD'):
    for electrode in ('TE','BE'):
        lbl = f'SX35{letter} top {electrode}'
        _df = df[df['notes_2.text'].str.contains(lbl)][-11:]

        x  = _df['keithley6430_1.v']
        y  = _df['keithley6430_1.i']

        (m,_),_,_ = linear_fit_n_variables(y, x)
        print(f'{lbl} {1/m/1e3:0.3f} kohm')
    
    lbl = f'SX35{letter} btm heater'
    _df = df[df['notes_2.text'].str.contains(lbl)][-11:]

    x  = _df['keithley6430_1.v']
    y  = _df['keithley6430_1.i']

    (m,_),_,_ = linear_fit_n_variables(y, x)
    print(f'{lbl} {1/m/1e3:0.3f} kohm')

## 2026/05/01 HZOFeFETv4 SX22 DG devices after HF

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tpo_tools import twinx, fig2clipboard, linear_fit_n_variables
import time

In [ ]:
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=["r", "r", "b", "b", 'g','g'],
    linestyle=["--", "-", "--", "-", '-','-'],
)

df = pd.read_excel('2026-05-01\\2026-05-01_152522_measurement.xlsx')

# HZOFeFETv4 SX22M btm
# DG device, BG only, TG floating
# For top (btm) devices, C1(4)->2602A, C2(3)->6430, C4(1)->ground (shorting cap)
# Using ESD protection box

for letter in 'ABCDEFGHIJKLMNOP':
    fig,axes = plt.subplots(2,2,figsize=(9,5))
    axes:list[list[plt.Axes]]
    
    axes[0,0].set_title(f'HZOFeFETv4 SX22{letter}, floating top gate')
    axes[0,1].set_title(f'HZOFeFETv4 SX22{letter}, floating bottom gate')
    for ax in axes.reshape(-1):
        ax.set_ylim(9e-13, 2e-6)
        ax.set_xlim(-1.8,1.8)
    axes[1,0].set_xlabel('$V_g$ (V)')
    axes[1,1].set_xlabel('$V_g$ (V)')
    axes[0,0].set_ylabel('$I$ (A), top device')
    axes[1,0].set_ylabel('$I$ (A), btm device')

    for device,row_of_axes in zip(('top','btm'),axes):
        for gate,ax in zip(('BG','TG'),row_of_axes):
            _df =  df[ df['notes_3.text'].str.contains(f'SX22{letter} {device}')]
            _df = _df[_df['notes_3.text'].str.contains(f'{gate} only')]

            _df = _df[-140:]

            x = _df['keithley6430_2.v'].to_numpy()
            y = _df['keithley2602_1.smua.i'].to_numpy()
            y2 = _df['keithley6430_2.i'].to_numpy()

            x2 = _df['keithley2602_1.smua.setv'].to_numpy()
            x -= np.mean(x[x2 == 0])
            y -= np.mean(y[x2 == 0])
            y2 -= np.mean(y2[x2 == 0])

            x = x[x2 > 0]
            y = y[x2 > 0]
            y2 = y2[x2 > 0]
        
            n = len(x)
            mid = n // 2

            X = np.full((n, 4), np.nan)
            Y = np.full((n, 4), np.nan)

            col = 0
            for s, e in [(0, mid), (mid, n)]:
                q = s + max(1, (e - s) // 4)  # first quarter endpoint
                X[s:q, col] = x[s:q]
                Y[s:q, col] = y[s:q]
                col += 1
                X[q:e, col] = x[q:e]
                Y[q:e, col] = y[q:e]
                col += 1

            ax.plot(X,Y, label=['1st $I_d$, 1st 1/4',
                                '1st $I_d$, remainder',
                                '2nd $I_d$, 1st 1/4',
                                '2nd $I_d$, remainder'])
            ax.axhline(y=1e-10, c='g', ls='--', label='EBIC TIA max')
            ax.plot(x,np.abs(y2), label='$|I_g|$')
            ax.set_yscale('log')
            ax.grid()

            if 'TG' in gate:
                ax.set_ylabel('')
                ax.set_yticklabels([])
            if 'top' in device:
                ax.set_xlabel('')
                ax.set_xticklabels([])
    axes[0,0].legend(loc='upper left')
    fig.tight_layout()
    fig2clipboard(fig, 350)
    print(letter)
    time.sleep(1)
    plt.close()
    # break

## 2026/05/04 2D Sweeps on DG FeFETs SX22

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tpo_tools import twinx, fig2clipboard, figures2gif, linear_fit_n_variables
import time

### HZOFeFETv4 SX22C btm

In [ ]:
df = pd.read_excel('2026-05-04\\2026-05-04_152903_measurement.xlsx')
df = df[df['notes_1.text'].str.contains('SX22C btm')]
df.reset_index()

masks = []
change_indices_all = df.index[df['keithley2602_2.smub.setv'].ne(df['keithley2602_2.smub.setv'].shift())] # can't just use unique because there are repeats

drain_curr_offset = df.loc[
    (df['keithley2602_2.smua.setv'] == 0) &
    (df['keithley2602_2.smub.setv'] == 0) &
    (df['keithley6430_3.setv'] == 0),
    'keithley2602_2.smua.i'
].mean()
print(drain_curr_offset*1e9, 'nA')

In [ ]:
# Match the entire array pattern to contiguous sections in df
array = np.concatenate([
    np.arange(0, 1.6, 0.1),
    np.arange(1.6, -1.6, -0.1),
    np.arange(-1.6, 0, 0.1)
])
fig,ax = plt.subplots()
ax.plot(array)
ax.grid()

sub_dfs = []

# Find all contiguous sections matching the array pattern
for i in range(len(df) - len(array) + 1):
    section = df['keithley6430_3.setv'].iloc[i:i+len(array)].values
    if np.allclose(section, array, atol=1e-6):
        sub_dfs.append( df.iloc[i:i+len(array)] )

In [ ]:
def get_IV(df):
    V_TG = df['keithley2602_2.smub.setv'].unique()[0]
    bg_bias = df['keithley6430_3.setv']
    drain_curr = (df['keithley2602_2.smua.i'] - drain_curr_offset)*1e6
    return V_TG, bg_bias, drain_curr

In [ ]:
fig,ax = plt.subplots()

ax.set_title('Floating TG')
ax.set_ylabel('current (uA)')
ax.set_xlabel('BG bias (V)')
ax.set_xlim(-1.8,1.8)
ax.set_ylim(0.5,5)
ax.grid()

for df_ in sub_dfs:
    if not len(df_[df_['notes_1.text'].str.contains('FLOAT')]):
        continue
    V_TG, bg_bias, drain_curr = get_IV(df_)
    ax.plot(bg_bias,np.abs(drain_curr))
fig.tight_layout()

In [ ]:
figs = []

for df_ in sub_dfs:
    V_TG, bg_bias, drain_curr = get_IV(df_)

    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_ylabel('current (uA)')
    ax.set_xlabel('BG bias (V)')
    ax.set_title(f'TG bias = {V_TG} V')

    ax.plot(bg_bias,drain_curr, label=V_TG)

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0.5,2.5)
    ax.grid()

    fig.tight_layout()
    # break

figures2gif(figs, './2026-05-04_SX22C-btm_2DSweep.gif')

Sweep 1 and sweep 2 comparison

In [ ]:
figs = []
n = len(sub_dfs)

for df1, df2, df3 in zip(sub_dfs[:n//3], 
                         sub_dfs[n//3:2*n//3], 
                         sub_dfs[2*n//3:]):
    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_ylabel('current (uA)')
    ax.set_xlabel('BG bias (V)')

    for j,df_ in enumerate((df1,df2,df3)):
        V_TG, bg_bias, drain_curr = get_IV(df_)
        ax.plot(bg_bias, np.abs(drain_curr), label=f'Sweep {j+1}, $V_{{TG}}$={V_TG}V')

    ax.set_title(f'TG bias = {V_TG} V')

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0.5,2.5)
    ax.grid()
    ax.legend(loc='upper left')

    fig.tight_layout()
    # break

figures2gif(figs, './2026-05-04_SX22C-btm_2DSweeps-comp.gif')

In [ ]:
figs = []
start = -9

for i in range(100):
    df_ = df.iloc[start-64:start-1]
    if len(df_) == 0:
        print(i, start)
        break
    elif df_['keithley6430_3.setv'].max() < 1.6:
        continue
    start -= 64

    V_TG = df_['keithley2602_2.smub.setv'].unique()
    bg_bias = df_['keithley6430_3.setv']
    drain_curr = (df_['keithley2602_2.smua.i'] - drain_curr_offset)*1e6

    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_title(V_TG)

    ax.plot(bg_bias,drain_curr, label=V_TG)

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0.5,2.5)
    ax.grid()

    fig.tight_layout()
    # break

figures2gif(figs[::-1], './2026-05-04_SX22C-btm_2DSweep.gif')

#### Sweep 1

In [ ]:
change_indices = change_indices_all[:len(change_indices_all)//2]

figs = []
for start, stop in zip(change_indices[:-1], change_indices[1:]):
    df_ = df.loc[start:stop-1]
    V_TG = df_['keithley2602_2.smub.setv'].unique()[0]
    x = df_['keithley6430_3.setv']
    y = df_['keithley2602_2.smua.i'] - drain_curr_offset

    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_title(V_TG)

    ax.plot(x,y, label=V_TG)

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0.5e-6,2.25e-6)
    ax.grid()

    fig.tight_layout()
    # break
figures2gif(figs, './2026-05-04_SX22C-btm_2DSweep1.gif')

#### Sweep 2

In [ ]:
# change_indices = change_indices_all[len(change_indices_all)//2:]
figs = []
start = 0
while True:
    try:
        df_ = df.loc[start:start+63]
    except:
        break
    finally:
        if len(df_) == 0:
            break
    start += 63
    V_TG = df_['keithley2602_2.smub.setv'].unique()
    bg_bias = df_['keithley6430_3.setv']
    drain_curr = (df_['keithley2602_2.smua.i'] - drain_curr_offset)*1e6

    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_title(V_TG)

    ax.plot(bg_bias,drain_curr, label=V_TG)

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0.5,2.5)
    ax.grid()

    fig.tight_layout()
    # break

figures2gif(figs, './2026-05-04_SX22C-btm_2DSweep.gif')

## 2026/05/05 2D Sweeps on DG FeFETs SX22

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tpo_tools import twinx, fig2clipboard, figures2gif, linear_fit_n_variables
import time
date = '2026-05-05'

In [ ]:
def get_IV(df):
    V_TG = df['keithley2602_2.smub.setv'].unique()[0]
    bg_bias = df['keithley6430_3.setv']
    drain_curr = (df['keithley2602_2.smua.i'] - drain_curr_offset)*1e6
    return V_TG, bg_bias, drain_curr

### HZOFeFETv4 SX22H top

In [ ]:
sample = 'SX22H-top'
df = pd.read_excel(f'./{date}/{date}_130044_measurement.xlsx')
df = df[df['notes_1.text'].str.contains(sample.replace('-', ' '))]
df.reset_index()

masks = []
change_indices_all = df.index[df['keithley2602_2.smub.setv'].ne(df['keithley2602_2.smub.setv'].shift())] # can't just use unique because there are repeats

drain_curr_offset = df.loc[
    (df['keithley2602_2.smua.setv'] == 0) &
    (df['keithley2602_2.smub.setv'] == 0) &
    (df['keithley6430_3.setv'] == 0),
    'keithley2602_2.smua.i'
].mean()
print(drain_curr_offset*1e9, 'nA')

In [ ]:
# Match the entire array pattern to contiguous sections in df
array = np.concatenate([
    np.arange(0, 1.6, 0.1),
    np.arange(1.6, -1.6, -0.1),
    np.arange(-1.6, 0.1, 0.1)
])
fig,ax = plt.subplots()
ax.plot(array)
ax.grid()

# Find all contiguous sections matching the array pattern
sub_dfs = []
for i in range(1, len(df) - len(array) + 1): # skip first instance
    section = df['keithley6430_3.setv'].iloc[i:i+len(array)].values
    if np.allclose(section, array, atol=1e-6):
        sub_dfs.append( df.iloc[i:i+len(array)] )

print(len(sub_dfs))

In [ ]:
fig,ax = plt.subplots()
figs.append(fig)
ax.set_ylabel('current (uA)')
ax.set_xlabel('BG bias (V)')
ax.set_title(f'Floating gate')
ax.set_xlim(-1.8,1.8)
ax.set_ylim(0,2)
ax.grid()

for i, df_ in enumerate(sub_dfs):
    V_TG, bg_bias, drain_curr = get_IV(df_)
    label = ''
    if df_['notes_1.text'].str.contains('TG is FLOAT').any():
        label = 'TG floating'
    elif df_['notes_1.text'].str.contains('BG is FLOAT').any():
        label = 'BG floating'
    else:
        break

    ax.plot(bg_bias,drain_curr, label=label)

ax.legend()
fig.tight_layout()
fig2clipboard(fig, 350)

In [ ]:
figs = []

for i, df_ in enumerate(sub_dfs):
    V_TG, bg_bias, drain_curr = get_IV(df_)
    if i == 0:
        V_TG = 'float'
    elif i in (1,2):
        continue

    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_ylabel('current (uA)')
    ax.set_xlabel('BG bias (V)')
    ax.set_title(f'TG bias = {V_TG} V')

    ax.plot(bg_bias,drain_curr, label=V_TG)

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0,5)
    ax.grid()

    fig.tight_layout()
    # break

figures2gif(figs, f'./{date}/{date}_{sample}_2DSweep.gif')

Sweep 1 and sweep 2 comparison

In [ ]:
figs = []
skip = 3 # first three are not part of the sweeps
n = len(sub_dfs[skip:]) 

for df1, df2 in zip(sub_dfs[skip:][:n//2 ], 
                    sub_dfs[skip:][ n//2:]):
    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_ylabel('current (uA)')
    ax.set_xlabel('BG bias (V)')

    for j,df_ in enumerate((df1,df2)):
        V_TG, bg_bias, drain_curr = get_IV(df_)
        ax.plot(bg_bias, np.abs(drain_curr), label=f'Sweep {j+1}, $V_{{TG}}$={V_TG}V')

    ax.set_title(f'TG bias = {V_TG} V')

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0.,5)
    ax.grid()
    ax.legend(loc='upper left')

    fig.tight_layout()
    # break

figures2gif(figs, f'./{date}/{date}_{sample}_2DSweeps-comp.gif')

## 2026/05/06 SX22F top

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tpo_tools import twinx, fig2clipboard, figures2gif, linear_fit_n_variables
import time
date = '2026-05-06'

In [ ]:
def get_IV(df):
    V_TG = df['keithley2602_2.smub.setv'].unique()[0]
    bg_bias = df['keithley6430_3.v']
    drain_curr = (df['keithley2602_2.smua.i'] - drain_curr_offset)
    gate_curr = (df['keithley6430_3.i'])
    return V_TG, bg_bias, drain_curr, gate_curr

### HZOFeFETv4 SX22F top

In [ ]:
sample = 'SX22F top'
df = pd.read_excel(f'./{date}/{date}_measurement.xlsx')
print(df['notes_1.text'].unique())
df = df[df['notes_1.text'].str.contains(sample)]
df.reset_index()

masks = []
change_indices_all = df.index[df['keithley2602_2.smub.setv'].ne(df['keithley2602_2.smub.setv'].shift())] # can't just use unique because there are repeats

drain_curr_offset = df.loc[
    (df['keithley2602_2.smua.setv'] == 0) &
    (df['keithley2602_2.smub.setv'] == 0) &
    (df['keithley6430_3.setv'] == 0),
    'keithley2602_2.smua.i'
].mean()
drain_curr_offset = 0
print(drain_curr_offset*1e9, 'nA')

In [ ]:
# Match the entire array pattern to contiguous sections in df
array = np.concatenate([
    np.arange(0, 1.7, 0.1),
    np.arange(1.7, -1.7, -0.1),
    np.arange(-1.7, 0.1, 0.1)
])
fig,ax = plt.subplots()
ax.plot(array)
ax.grid()

# Find all contiguous sections matching the array pattern
sub_dfs = []
for i in range(1, len(df) - len(array) + 1): # skip first instance
    section = df['keithley6430_3.setv'].iloc[i:i+len(array)].values
    if np.allclose(section, array, atol=1e-6):
        sub_dfs.append( df.iloc[i:i+len(array)] )

print(len(sub_dfs))

In [ ]:
fig,ax = plt.subplots()
figs.append(fig)
ax.set_ylabel('|current| (A)')
ax.set_xlabel('BG bias (V)')
ax.set_title(f'Floating gate')
ax.set_xlim(-1.8,1.8)
# ax.set_ylim(0,0.6)
ax.set_yscale('log')
ax.grid()

for i, df_ in enumerate(sub_dfs):
    V_TG, bg_bias, drain_curr, gate_curr = get_IV(df_)
    label = ''
    if df_['notes_1.text'].str.contains('TG is FLOAT').any():
        label = 'TG floating'
    elif df_['notes_1.text'].str.contains('BG is FLOAT').any():
        label = 'BG floating'
    else:
        break

    ax.plot(bg_bias,np.abs(drain_curr), label=label, color=f'C{i}')
    ax.plot(bg_bias,np.abs(gate_curr), color=f'C{i}')

ax.legend()
fig.tight_layout()
fig2clipboard(fig, 350)

In [ ]:
figs = []

for i, df_ in enumerate(sub_dfs):
    V_TG, bg_bias, drain_curr = get_IV(df_)
    if i == 0:
        V_TG = 'float'
    elif i in (1,2):
        continue

    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_ylabel('current (uA)')
    ax.set_xlabel('BG bias (V)')
    ax.set_title(f'TG bias = {V_TG} V')

    ax.plot(bg_bias,drain_curr, label=V_TG)

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0,5)
    ax.grid()

    fig.tight_layout()
    # break

figures2gif(figs, f'./{date}/{date}_{sample}_2DSweep.gif')

## SX14 DG FeFETs (tungsten, v3)

In [ ]:
date = '2026-05-07'
df = pd.read_excel(f'./{date}/{date}_184013_measurement.xlsx')
df.reset_index()

masks = []
change_indices_all = df.index[df['keithley2602_2.smub.setv'].ne(df['keithley2602_2.smub.setv'].shift())] # can't just use unique because there are repeats

drain_curr_offset = df.loc[
    (df['keithley2602_2.smua.setv'] == 0) &
    (df['keithley2602_2.smub.setv'] == 0) &
    (df['keithley6430_3.setv'] == 0),
    'keithley2602_2.smua.i'
].mean()
drain_curr_offset = 0
print(drain_curr_offset*1e9, 'nA')

In [ ]:
# Match the entire array pattern to contiguous sections in df
array = np.concatenate([
    np.arange(0, 1.7, 0.1),
    np.arange(1.7, -1.7, -0.1),
    np.arange(-1.7, 0.1, 0.1)
])
fig,ax = plt.subplots()
ax.plot(array)
ax.grid()

# Find all contiguous sections matching the array pattern
sub_dfs = []
for i in range(1, len(df) - len(array) + 1): # skip first instance
    section = df['keithley6430_3.setv'].iloc[i:i+len(array)].values
    if np.allclose(section, array, atol=1e-6):
        sub_dfs.append( df.iloc[i:i+len(array)] )

print(len(sub_dfs))

In [ ]:
for letter in 'ABCDEFGHIJKLMNOP':
    for tb in ('top', 'btm'):
        fig,ax = plt.subplots()
        figs.append(fig)
        ax.set_ylabel('|current| (A)')
        ax.set_xlabel('BG bias (V)')
        ax.set_title(f'SX14{letter} {tb}, floating gate')
        ax.set_xlim(-1.8,1.8)
        # ax.set_ylim(0,0.6)
        ax.set_yscale('log')
        ax.grid()

        i = 0
        for df in sub_dfs:
            if f'SX14{letter} {tb}' in df['notes_1.text'].iloc[0]:
                V_TG, bg_bias, drain_curr, gate_curr = get_IV(df)
                label = ''
                if df['notes_1.text'].str.contains('TG is FLOAT').any():
                    label = 'TG floating'
                elif df['notes_1.text'].str.contains('BG is FLOAT').any():
                    label = 'BG floating'
                else:
                    break

                ax.plot(bg_bias,np.abs(drain_curr), label=label, color=f'C{i}')
                ax.plot(bg_bias,np.abs(gate_curr), color=f'C{i}')
                i += 1
        if i == 0:
            plt.close()
            continue

        ax.legend()
        fig.tight_layout()
        fig2clipboard(fig, 350)
        print(letter, tb)
        # break
        time.sleep(1)

In [ ]:
figs = []

for i, df_ in enumerate(sub_dfs):
    V_TG, bg_bias, drain_curr = get_IV(df_)
    if i == 0:
        V_TG = 'float'
    elif i in (1,2):
        continue

    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_ylabel('current (uA)')
    ax.set_xlabel('BG bias (V)')
    ax.set_title(f'TG bias = {V_TG} V')

    ax.plot(bg_bias,drain_curr, label=V_TG)

    ax.set_xlim(-1.8,1.8)
    ax.set_ylim(0,5)
    ax.grid()

    fig.tight_layout()
    # break

figures2gif(figs, f'./{date}/{date}_{sample}_2DSweep.gif')

# 2026/05/27 HZOFeFETv4 SX22F top (DG)

In [ ]:
date = '2026-05-27'
# df = pd.read_excel(f'./{date}/{date}_184013_measurement.xlsx')
df = pd.read_csv(f'../temp\\{date}\\{date}_121256_raw.csv')
df.reset_index()

masks = []

drain_curr_offset = df.loc[
    (df['keithley2602_2.smua.setv'] == 0) &
    (df['keithley6430_3.setv'] == 0),
    'keithley2602_2.smua.i'
].mean()
drain_curr_offset = 0
print(drain_curr_offset*1e9, 'nA')

In [ ]:
# Match the entire array pattern to contiguous sections in df
array = np.concatenate([
    np.arange(0, 1.8, 0.1),
    np.arange(1.8, -1.8, -0.1),
    np.arange(-1.8, 0.1, 0.1)
])
fig,ax = plt.subplots()
ax.plot(array)
ax.grid()

# Find all contiguous sections matching the array pattern
sub_dfs = []
for i in range(1, len(df) - len(array) + 1): # skip first instance
    section = df['keithley6430_3.setv'].iloc[i:i+len(array)].values
    if np.allclose(section, array, atol=1e-6):
        sub_dfs.append( df.iloc[i:i+len(array)] )

print(len(sub_dfs))

In [ ]:
figs = []

for i, df_ in enumerate(sub_dfs):
    V_TG, bg_bias, drain_curr, gate_curr = get_IV(df_)
    if i == 0:
        V_TG = 'float'
    elif i in (1,2):
        continue

    fig,ax = plt.subplots()
    figs.append(fig)
    ax.set_ylabel('current (A)')
    ax.set_xlabel('BG bias (V)')
    ax.set_title(f'TG bias = {V_TG} V')

    ax.plot(bg_bias,np.abs(drain_curr), label=V_TG)

    # ax.set_xlim(-1.8,1.8)
    ax.set_ylim(1e-10,1e-6)
    ax.set_yscale('log')
    ax.grid()

    fig.tight_layout()
    break

# figures2gif(figs, f'./{date}/{date}_{sample}_2DSweep.gif')

# 2026/05/29 MoS2FETv1 SX28*

In [ ]:
date = '2026-05-29'
df = pd.read_excel(f'./{date}/{date}_150443_measurement.xlsx')

df.reset_index()

masks = []

drain_curr_offset = df.loc[
    (df['keithley2602_2.smua.setv'] == 0) &
    (df['keithley6430_3.setv'] == 0),
    'keithley2602_2.smua.i'
].mean()
drain_curr_offset = 0
print(drain_curr_offset*1e9, 'nA')

In [ ]:
desc = """MoS2FETv1 SX28G btm / D2
C4 = G = 6430
C3 = D = 2602A
C2 = S = ground"""

df = df[df['notes_1.text'].str.contains(desc)]

V_TG, bg_bias, drain_curr, gate_curr = get_IV(df[-77:])
fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(bg_bias, drain_curr, label='drain current')
ax.plot(bg_bias, gate_curr, label='gate current')

ax.set_xlabel('gate bias (V)')
ax.set_ylabel('current (A)')
# ax.set_yscale('log')
ax.grid(True)
ax.legend()

fig.tight_layout()
fig2clipboard(fig, 350)

# 2026/09/28 HZOFeFETv5 SX29

In [ ]:
date = '2026-10-02'
# df = pd.read_excel(f'./{date}/{date}_163107_measurement.xlsx')
# df = pd.read_excel(f'./{date}/{date}_185646_measurement.xlsx')
df = pd.read_excel(f'./{date}/{date}_082536_measurement.xlsx')
df.reset_index()

masks = []

notesidx = 1
k2602idx = 2

drain_curr_offset = df.loc[
    (df[f'keithley2602_{k2602idx}.smua.setv'] == 0) &
    (df[f'keithley2602_{k2602idx}.smua.setv'] == 0),
    f'keithley2602_{k2602idx}.smub.i'
].mean()
drain_curr_offset = 0
print(drain_curr_offset*1e9, 'nA')

def get_iv(df):
    bg_bias = df[f'keithley2602_{k2602idx}.smua.v']
    drain_curr = (df[f'keithley2602_{k2602idx}.smub.i'] - drain_curr_offset)
    gate_curr = (df[f'keithley2602_{k2602idx}.smua.i'])
    return bg_bias, drain_curr, gate_curr

In [ ]:
for i in range(16):
    letter = chr(65 + i)

    if letter not in 'C':
        continue
    fig, (ax1,ax2) = plt.subplots(2,1, figsize=(6, 6))
    ax = ax1
    for tb in ('top','btm'):
        ax.set_xlabel('gate bias (V)')
        ax.set_ylabel('current (A)')
        # ax.set_yscale('log')
        ax.set_xlim(-1.55, 2.55)

        ax.set_title(f'SX29{letter} {tb}')
        ax.grid(True)

        try:
            df2 = df[df[f'notes_{notesidx}.text'].str.contains(f'SX29{letter} {tb}')]
        except:
            continue

        bg_bias, drain_curr, gate_curr = get_iv(df2)

        ax.plot(bg_bias, drain_curr, label='D')
        ax.plot(bg_bias, gate_curr, label='G')

        ax = ax2

    ax1.legend()
    fig.tight_layout()

    fig2clipboard(fig, 350)
    # print(letter, end='\r')
    # time.sleep(3)
    # plt.close(fig)

# 2026/10/08 DG MOSFET (HZOFeFETv5) SX16

Did not measure prior to KOH

In [ ]:
import matplotlib as mpl
# df = pd.read_excel(f'./2026-10-08/2026-10-09_083744_measurement.xlsx') # after KOH
# df = pd.read_excel(f'./2026-10-09/2026-10-09_121235_measurement.xlsx') # after HF
# df = pd.read_excel(f'./2026-10-09/2026-10-09_163931_measurement.xlsx')# after dicing
# df = pd.read_excel(f'./2026-10-09/2026-10-10_153008_measurement.xlsx')# after dicing, all devices
df = pd.read_excel('./2026-10-09/2026-10-11_150357_measurement.xlsx') # more datapoints, higher Vds
df.reset_index()

notes = 'notes_1.text'
bg = 'keithley2602_2.smua'
tg = 'keithley6430_3'
d = 'keithley2602_2.smub'


tb = 'btm'
for letter in 'ABCDEFGHIJKLMNOP':
    fig, axes = plt.subplots(3,1, figsize=(7, 7))

    axd,axbg,axtg = axes
    axd.set_title(f'SX16{letter} {tb}')
    axd.set_ylabel('$I_d$ (A)')
    axbg.set_ylabel('$I_{bg}$ (A)')
    axtg.set_ylabel('$I_{tg}$ (A)')
    axd.set_yscale('log')
    axbg.set_yscale('symlog', linthresh=1e-13)
    axtg.set_yscale('symlog', linthresh=1e-13)
    for ax in axes:
        ax.grid()
        ax.set_xlim(-1.7, 1.7)

    axd2 = twinx(axd, color='k')
    axd2.set_ylabel('$I_d/V_{ds}$ ($\\Omega ^{-1}$)')
    axd2.set_yscale(axd.get_yscale())

    df2 = df[df[notes].str.contains(f'SX16{letter} {tb}')]
    if len(df2) == 0:
        continue

    axd.set_title(f'SX16{letter} {tb}, $V_{{ds}}$={df2[f'{d}.setv'].unique()[1]}V')

    cmap = mpl.colormaps['viridis']
    for TG_V in df2[f'{tg}.setv'].unique()[::-1]:
        df3 = df2[(df2[f'{tg}.setv'] == TG_V) & (df2[f'{d}.setv'] != 0)]
        if len(df3) < 10:
            continue
        bg_bias = df3[f'{bg}.v']
        bg_curr = df3[f'{bg}.i']
        tg_bias = df3[f'{tg}.v']
        tg_curr = df3[f'{tg}.i']
        d_volt = df3[f'{d}.v']
        d_curr = df3[f'{d}.i']

        color = cmap((TG_V - df2[f'{tg}.setv'].min()) / 
                        (df2[f'{tg}.setv'].max() - df2[f'{tg}.setv'].min()))
        axd.plot(df3[f'{bg}.v'], df3[f'{d}.i'], label=f'TG={TG_V}V', color=color)
        axbg.plot(df3[f'{bg}.v'], df3[f'{bg}.i'], label=f'TG={TG_V}V', color=color)
        axtg.plot(df3[f'{bg}.v'][3:-3], df3[f'{tg}.i'][3:-3], label=f'TG={TG_V}V', color=color) # capacitive coupling at voltage steps

    axd.legend()
    axd2.set_ylim([l/df2[f'{d}.setv'].max() for l in axd.get_ylim()])
    fig.tight_layout()
    # break
    # fig2clipboard(fig, 350)
    print(letter, end='\r')
    # time.sleep(1)
    # plt.close()

In [ ]:
letters = 'ABCDEFGHIJKLMNOP'
letters = letters[::4] + letters[1::4] + letters[2::4] + letters[3::4]

titles = ['LxW 100nmx200nm', 'LxW 200nmx200nm', 'LxW 500nmx200nm', 'LxW 1000nmx200nm']

for i,letter in enumerate(letters):
    if i % 4 == 0:
        fig,(ax, ax2) = plt.subplots(2,1,figsize=(4,6))
    for tb in ('top','btm'):
        df2 = df[df[notes].str.contains(f'SX16{letter} {tb}')]

        I_THRESHOLD = 1e-9  # A

        xx,yy = [],[]
        ion = []

        for TG_V in df2[f'{tg}.setv'].unique():
            if TG_V == 0:
                continue
            df3 = df2[df2[f'{tg}.setv'] == TG_V].copy()
            ion.append( df3[f'{d}.i'].max() )

            x = df3[f'{bg}.v'].to_numpy()
            y = np.abs(df3[f'{d}.i'].to_numpy())

            order = np.argsort(x)
            x, y = x[order], y[order]

            crossings = np.where((y[:-1] - I_THRESHOLD) * (y[1:] - I_THRESHOLD) <= 0)[0]
            vth = []

            for c in crossings:
                v = np.interp(I_THRESHOLD, [y[c], y[c + 1]], [x[c], x[c + 1]])
                vth.append(v)

            # print(f'SX16{letter} {tb}, TG = {TG_V:.3g} V: '
            #         f'Vth = {[f"{v:.3g}" for v in vth] if vth else "not found"}')

            xx.append(TG_V)
            yy.append(np.mean(vth))

        fmts = {'top': '-o', 'btm': '--x'}
        ax.plot(xx,yy, fmts[tb], label=f'{letter} {tb}', color=f'C{i%4}')
        ax2.plot(xx,ion, fmts[tb], label=f'{letter} {tb}', color=f'C{i%4}')
        if i % 4 == 3:
            ax2.set_xlabel('$V_{TG}$ (V)')
            ax1.set_xticklabels([])
            ax.set_ylabel('$V_{BG,th}$ (V)')
            ax2.set_ylabel('$I_{on}$ (A)')
            ax.set_xlim(-1.7,1.7)
            ax2.set_xlim(-1.7,1.7)
            ax.set_ylim(-1.1,1.1)
            ax2.set_ylim(1e-8,7e-7)
            ax2.set_yscale('log')
            ax.grid(alpha=0.3)
            ax2.grid(which='both', alpha=0.3)
            ax.legend(ncol=2)
            ax.set_title(titles[i//4])
            fig.tight_layout()

## Sweeps with more data

In [ ]:
df = pd.read_excel(f'./2026-10-09/2026-10-11_120802_measurement.xlsx')# after dicing, all devices
df.reset_index()

notes = 'notes_1.text'
bg = 'keithley2602_2.smua'
tg = 'keithley6430_3'
d = 'keithley2602_2.smub'


for letter in 'ABCDEFGHIJKLMNOP':
    for tb in ('top', 'btm'):
        samp = f'SX16{letter} {tb}'
        df2 = df[df[notes].str.contains(samp)]
        if len(df2) == 0:
            continue

        fig, axes = plt.subplots(3,1, figsize=(7, 7))

        axd,axbg,axtg = axes
        axd.set_title(samp)
        axd.set_ylabel('$I_d$ (A)')
        axbg.set_ylabel('$I_{bg}$ (A)')
        axtg.set_ylabel('$I_{tg}$ (A)')
        # axd.set_yscale('log')
        # axbg.set_yscale('symlog', linthresh=1e-13)
        # axtg.set_yscale('symlog', linthresh=1e-13)
        # for ax in axes:
            # ax.grid()
            # ax.set_xlim(-1.7, 1.7)

        axd2 = twinx(axd, color='k')
        axd2.set_ylabel('$I_d/V_{ds}$ ($\\Omega ^{-1}$)')
        axd2.set_yscale(axd.get_yscale())


        axd.set_title(f'{samp}, $V_{{ds}}$={df2[f'{d}.setv'].unique()[1]}V')

        cmap = mpl.colormaps['viridis']
        axd.tripcolor(df2[f'{bg}.v'], df2[f'{tg}.v'], df2[f'{d}.i'])
        # for TG_V in df2[f'{tg}.setv'].unique()[::-1]:
        #     df3 = df2[(df2[f'{tg}.setv'] == TG_V) & (df2[f'{d}.setv'] != 0)]
        #     if len(df3) < 10:
        #         continue
        #     bg_bias = df3[f'{bg}.v']
        #     bg_curr = df3[f'{bg}.i']
        #     tg_bias = df3[f'{tg}.v']
        #     tg_curr = df3[f'{tg}.i']
        #     d_volt = df3[f'{d}.v']
        #     d_curr = df3[f'{d}.i']

        #     color = cmap((TG_V - df2[f'{tg}.setv'].min()) / 
        #                     (df2[f'{tg}.setv'].max() - df2[f'{tg}.setv'].min()))
        #     axbg.plot(df3[f'{bg}.v'], df3[f'{bg}.i'], label=f'TG={TG_V}V', color=color)
        #     axtg.plot(df3[f'{bg}.v'][3:-3], df3[f'{tg}.i'][3:-3], label=f'TG={TG_V}V', color=color) # capacitive coupling at voltage steps

        # axd.legend()
        # axd2.set_ylim([l/df2[f'{d}.setv'].max() for l in axd.get_ylim()])
        fig.tight_layout()
        # break
        # fig2clipboard(fig, 350)
        # print(letter, end='\r')
        # time.sleep(1)
        # plt.close()

# 2026/10/22 DG FeSFET (HZOFeFETv5) SX30, SX35

Prior to KOH

In [ ]:
import matplotlib as mpl
# sample, df = 'SX30', pd.read_excel(f'./2026-10-21/2026-10-22_114833_measurement.xlsx') # SX30 before KOH (as-received)
# sample, df = 'SX35', pd.read_excel(f'./2026-10-21/2026-10-22_134006_measurement.xlsx') # SX35 after KOH
# sample, df = 'SX30', pd.read_excel(f'./2026-10-23/2026-10-23_180310_measurement.xlsx') # SX30 after KOH
# sample, df = 'SX30', pd.read_excel(f'./2026-10-23/2026-10-23_184904_measurement.xlsx') # SX30 after HF + dicing
sample, df = 'SX35', pd.read_excel(f'./2026-10-23/2026-10-23_192118_measurement.xlsx') # SX35 after HF + dicing


df.reset_index()
notes = 'notes_1.text'
bg = 'keithley2602_2.smua'
tg = 'keithley6430_3'
d = 'keithley2602_2.smub'


for letter in 'ABCDEFGHIJKLMNOP':
    fig, axes = plt.subplots(3,2, figsize=(9, 7))
    for axis in axes.ravel():
        axis.grid()
        axis.set_xlim(-1.7, 1.7)

    for ax,tb in zip(axes.T, ('top','btm')):

        axd,axbg,axtg = ax
        axd.set_title(f'{sample}{letter} {tb}')
        axd.set_ylabel('$I_d$ (A)')
        axbg.set_ylabel('$I_{bg}$ (A)')
        axtg.set_ylabel('$I_{tg}$ (A)')
        axd.set_yscale('log')
        axbg.set_yscale('symlog', linthresh=1e-13)
        axtg.set_yscale('symlog', linthresh=1e-13)

        axd2 = twinx(axd, color='k')
        axd2.set_ylabel('$I_d/V_{ds}$ ($\\Omega ^{-1}$)')
        axd2.set_yscale(axd.get_yscale())

        df2 = df[df[notes].str.contains(f'{sample}{letter} {tb}')]
        if len(df2) == 0:
            continue

        axd.set_title(f'{sample}{letter} {tb}, $V_{{ds}}$={df2[f'{d}.setv'].unique()[1]}V')

        cmap = mpl.colormaps['viridis']
        for TG_V in np.arange(1.5, -1.6, -0.5):
            df3 = df2[(df2[f'{tg}.setv'] == TG_V) & (df2[f'{d}.setv'] != 0)]
            if len(df3) < 10:
                continue
            bg_bias = df3[f'{bg}.v']
            bg_curr = df3[f'{bg}.i']
            tg_bias = df3[f'{tg}.v']
            tg_curr = df3[f'{tg}.i']
            d_volt = df3[f'{d}.v']
            d_curr = df3[f'{d}.i']

            color = cmap((TG_V - df2[f'{tg}.setv'].min()) / 
                         (df2[f'{tg}.setv'].max() - df2[f'{tg}.setv'].min()))
            axd.plot(df3[f'{bg}.v'], df3[f'{d}.i'], label=f'TG={TG_V}V', color=color)
            axbg.plot(df3[f'{bg}.v'], df3[f'{bg}.i'], label=f'TG={TG_V}V', color=color)
            axtg.plot(df3[f'{bg}.v'][3:-3], df3[f'{tg}.i'][3:-3], label=f'TG={TG_V}V', color=color) # capacitive coupling at voltage steps

        axd.legend()
        axd2.set_ylim([l/df2[f'{d}.setv'].max() for l in axd.get_ylim()])
    fig.tight_layout()
    # break
    fig2clipboard(fig, 350)
    print(letter, end='\r')
    time.sleep(0.5)
    plt.close()

In [ ]:
letters = 'ABCDEFGHIJKLMNOP'
letters = letters[::4] + letters[1::4] + letters[2::4] + letters[3::4]

titles = ['LxW 100nmx200nm', 'LxW 200nmx200nm', 'LxW 500nmx200nm', 'LxW 1000nmx200nm']

for i,letter in enumerate(letters):
    if i % 4 == 0:
        fig,(ax, ax2) = plt.subplots(2,1,figsize=(4,6))
    for tb in ('top','btm'):
        df2 = df[df[notes].str.contains(f'SX16{letter} {tb}')]

        I_THRESHOLD = 1e-9  # A

        xx,yy = [],[]
        ion = []

        for TG_V in df2[f'{tg}.setv'].unique():
            if TG_V == 0:
                continue
            df3 = df2[df2[f'{tg}.setv'] == TG_V].copy()
            ion.append( df3[f'{d}.i'].max() )

            x = df3[f'{bg}.v'].to_numpy()
            y = np.abs(df3[f'{d}.i'].to_numpy())

            order = np.argsort(x)
            x, y = x[order], y[order]

            crossings = np.where((y[:-1] - I_THRESHOLD) * (y[1:] - I_THRESHOLD) <= 0)[0]
            vth = []

            for c in crossings:
                v = np.interp(I_THRESHOLD, [y[c], y[c + 1]], [x[c], x[c + 1]])
                vth.append(v)

            # print(f'SX16{letter} {tb}, TG = {TG_V:.3g} V: '
            #         f'Vth = {[f"{v:.3g}" for v in vth] if vth else "not found"}')

            xx.append(TG_V)
            yy.append(np.mean(vth))

        fmts = {'top': '-o', 'btm': '--x'}
        ax.plot(xx,yy, fmts[tb], label=f'{letter} {tb}', color=f'C{i%4}')
        ax2.plot(xx,ion, fmts[tb], label=f'{letter} {tb}', color=f'C{i%4}')
        if i % 4 == 3:
            ax2.set_xlabel('$V_{TG}$ (V)')
            ax1.set_xticklabels([])
            ax.set_ylabel('$V_{BG,th}$ (V)')
            ax2.set_ylabel('$I_{on}$ (A)')
            ax.set_xlim(-1.7,1.7)
            ax2.set_xlim(-1.7,1.7)
            ax.set_ylim(-1.1,1.1)
            ax2.set_ylim(1e-8,7e-7)
            ax2.set_yscale('log')
            ax.grid(alpha=0.3)
            ax2.grid(which='both', alpha=0.3)
            ax.legend(ncol=2)
            ax.set_title(titles[i//4])
            fig.tight_layout()